# 05 · Transition states

A transition state has partial bonds that no force field types, so rxembed holds them while it embeds: as numbers
from a SMILES, or as a core of atoms copied from a TS geometry while the rest of the molecule is embedded anew. Each
render draws the held partial bonds dashed.

In [ ]:
import rxembed as rx
import xyzrender

## An SN2 TS from numbers

`fix=` holds the forming F...C and breaking C...Cl distances and the F-C-Cl angle in every conformer, and the
butyl chain is sampled around them. `rx.match` names the three atoms, and `hy=True` draws the hydrogens, which lie
flat about the reacting carbon.

In [ ]:
SN2 = "[F-].CCCCCl"  # fluoride and 1-chlorobutane
fluoride, carbon, chloride = rx.match(rx.parse_smiles(SN2), "[F-].[CH2]Cl")
held = {(fluoride, carbon): 2.02, (carbon, chloride): 2.28, (fluoride, carbon, chloride): 178.0}
sn2 = rx.embed(SN2, fix=held, n=4, seed=1).minimize()
for key in held:
    print(key, round(sn2.measure(key)["mean"], 2))

partial = [(fluoride + 1, carbon + 1), (carbon + 1, chloride + 1)]  # xyzrender counts atoms from 1
xyzrender.render(xyzrender.load(sn2.lowest(1).mol), ts_bonds=partial, hy=True)

## A TS core onto a new substrate

`rx.Template(ts, smarts)` copies the coordinates of the atoms a SMARTS matches in a TS geometry onto the same match
in another molecule. Here the F...C...Cl core of the fluoride and chloromethane TS goes onto isopropyl and allyl
chloride, and their carbon substituents are embedded around it.

In [ ]:
ts = rx.read_xyz("structures/sn2.xyz", charge=-1)  # fluoride and chloromethane at the TS
CORE = "[F].[#6]-[Cl]"
for substrate in ["[F-].CC(C)Cl", "[F-].C=CCCl"]:
    grafted = rx.embed(substrate, fix=rx.Template(ts, CORE), n=4, seed=1).minimize()
    fluoride, carbon, chloride = rx.match(grafted.mol, CORE)
    distances = [round(grafted.measure(pair)["mean"], 2) for pair in [(fluoride, carbon), (carbon, chloride)]]
    print(substrate, "F...C and C...Cl", distances)
    partial = [(fluoride + 1, carbon + 1), (carbon + 1, chloride + 1)]
    display(xyzrender.render(xyzrender.load(grafted.lowest(1).mol), ts_bonds=partial, hy=True))

## A computed TS onto a new catalyst

A computed TS retargets the same way. In this one a spiro beta-lactone closes as its C-N bond to morpholine breaks.
Written with pyrrolidine instead, and with piperidine on a 4-chlorophenyl substrate, the lactone ring, its carbonyl O
and the leaving N stay where the TS has them and the rest is embedded anew. Each render draws the new TS over the
original.

In [ ]:
ts = rx.read_xyz("structures/spiro-ts1.xyz", charge=0)
print(rx.smiles(ts))

CORE = "[#6]1-[#8]-[#6](-[#8-])(-[#7+])-[#6]-1"  # the lactone ring, its carbonyl O and the leaving N
VARIANTS = {
    "pyrrolidine": "CC1=NN(c2ccccc2)C(=O)[C@]12O[C@@]([O-])([NH+]1CCCC1)[C@@H]2c1ccccc1",
    "piperidine, 4-chlorophenyl": "CC1=NN(c2ccc(C(C)(C)(C))cc2)C(=O)[C@]12O[C@@]([O-])([NH+]1CCCCC1)[C@@H]2c1cc(OC)c(Cl)c(OC)c1",
}
for name, smiles in VARIANTS.items():
    retargeted = rx.embed(smiles, fix=rx.Template(ts, CORE), n=4, seed=1).minimize()
    core = rx.match(retargeted.mol, CORE)
    _, _, carbon, _, nitrogen, _ = core
    print(f"{name}: C...N {retargeted.measure((carbon, nitrogen))['mean']:.2f} A")
    breaking = [(carbon + 1, nitrogen + 1)]  # xyzrender counts atoms from 1
    display(xyzrender.render(xyzrender.load(retargeted.lowest(1).mol), overlay=xyzrender.load(ts), ts_bonds=breaking))